# Session 2: Standard scRNAseq pipeline
# Part 1

In [2]:
# —— RUN THIS BEFORE ANYTHING ELSE ———————————————————————————————
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    %run /content/drive/MyDrive/immerge_workshop/codes/notebooks/setup_notebook.py
else:
    os.chdir('/nfs/team292/projects/immerge_workshop')
    
os.getcwd()

'/nfs/team292/projects/immerge_workshop'

## Configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import gc
import glob as glob_mod
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import scvi
import seaborn as sns
from anndata.experimental import concat_on_disk

sc.settings.verbosity = 3
sc.set_figure_params(dpi=100, frameon=False, figsize=(6, 5))
ad.settings.allow_write_nullable_strings = True

# —— DIRECTORIES —————————————————————————————————————————————————
DATA_DIR = os.path.join("data/")
OUT_DIR = os.path.join("results/temp/session2_part1/")

os.makedirs(OUT_DIR, exist_ok=True)

# —— INPUT FILES —————————————————————————————————————————————————
DATASETS = {
    "Burns": os.path.join(DATA_DIR, "burns_f_2026.h5ad"),
    "Wang": os.path.join(DATA_DIR, "wang_f_2020.h5ad"),
}

## 1) Loading the datasets and concatenating
First we are going to load our datasets and concatenate them:

In [ ]:
# TO be removed
# —— READ EACH OBJECT INTO A DICTIONARY ——————————————————————————
adatas = {}

for k, v in DATASETS.items():
    print(f"Loading {k} dataset...")
    adatas[k] = sc.read_h5ad(v)
    print(adatas[k])
    
# —— CONCATENATE ALONG THE CELL AXIS —————————————————————————————
adata = ad.concat(
    list(adatas.values()),
    axis=0,
    join="inner"
)

## 2) Exploring the raw dataset

> ### ✏️ Exercise 1 — How many cells and genes do we have in our dataset?
>
> Print the **shape** of your gene expression matrix.
>

<details>
<summary><i>Show solution</i></summary>

```python
print(adata.shape)   
```
    
We have 61901 cells and 30926 genes.

</details>

Let's print the first 10 rows and columns of our gene expression matrix, do we have raw counts/integers on the matrix?

In [ ]:
adata.X[:10, :10].toarray()

What is the minimum number of counts we have in the matrix?

In [ ]:
adata.X.min()

And the maximum?

In [ ]:
adata.X.max()

Now let's make a histogram of all the expression values in our matrix. Which number is the most frequently found in the matrix?

In [ ]:
plt.hist(adata.X.data, bins=50, log=False)
plt.ticklabel_format(style='plain', axis='y')
plt.xlabel("Expression value")
plt.ylabel("Frequency")
plt.show()

Let's set log=True:

In [ ]:
plt.hist(adata.X.data, bins=50, log=True)
plt.xlabel("Expression value")
plt.ylabel("Frequency")
plt.show()

Now let's check the percentage of genes in each cell for which the gene expression value is different from 0:

In [ ]:
adata.obs["pct_genes_expressed"] = (adata.obs["n_genes"] / adata.n_vars) * 100

# One box per library
groups = adata.obs.groupby("Library_id", observed=True)["pct_genes_expressed"]
labels = list(groups.groups.keys())
data = [groups.get_group(g).values for g in labels]

fig, ax = plt.subplots(figsize=(max(6, 0.5 * len(labels)), 4))
ax.boxplot(data, labels=labels)
ax.set_ylabel("% Genes Expressed")
ax.set_xlabel("Library_id")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Median per library
adata.obs.groupby("Library_id", observed=True)["pct_genes_expressed"].median()

## 3) Quality control
We would like to plot:

- Number of genes expressed per cell
- Number of counts per cell
- Proportion of mitochondrial reads per cell
- Doublet scores

We first need to identify which genes are mitochondrial genes.

In [ ]:
# mitochondrial genes start with "MT-" for human
adata.var["mt"] = np.asarray(adata.var_names.str.startswith("MT-"), dtype=bool)
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, log1p=True)

Now let's plot these variables per library:

In [ ]:
# —— VIOLIN PLOTS PER LIBRARY ————————————————————————————————————
sc.pl.violin(
    adata,
    keys=["n_genes", "total_counts", "pct_counts_mt", "doublet_scores"],
    groupby="Library_id",
    jitter=0.4,
    multi_panel=True,
    rotation=90,
)

In [ ]:
MIN_GENES=1000
MAX_COUNTS=100000
MAX_PCT_MITO=20

# —— HISTOGRAMS WITH THRESHOLDS MARKED ———————————————————————————
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left panel: number of genes per cell.
sns.histplot(adata.obs["n_genes"], bins=100, ax=axes[0], color="#4C72B0")
axes[0].axvline(MIN_GENES, color="red", linestyle="--")
axes[0].set_title(f"Genes per cell (threshold = {MIN_GENES})")
axes[0].set_xlabel("n_genes")

# Mid panel: mitochondrial percentage per cell.
sns.histplot(adata.obs["pct_counts_mt"], bins=100, ax=axes[1], color="#DD8452")
axes[1].axvline(MAX_PCT_MITO, color="red", linestyle="--")
axes[1].set_title(f"% mitochondrial counts (threshold = {MAX_PCT_MITO})")
axes[1].set_xlabel("pct_counts_mt")

# Right panel: number of counts per cell.
sns.histplot(adata.obs["total_counts"], bins=100, ax=axes[2], color="#55A868")
axes[2].axvline(MAX_COUNTS, color="red", linestyle="--")
axes[2].set_title(f"Counts per cell (threshold = {MAX_COUNTS})")
axes[2].set_xlabel("total_counts")

plt.tight_layout()
plt.show()

What are good cutoffs?

In [ ]:
adata = adata[(adata.obs["n_genes"] >= MIN_GENES) & (adata.obs["pct_counts_mt"] < MAX_PCT_MITO) & (adata.obs["total_counts"] < MAX_COUNTS)]

> ### ✏️ Exercise 2 — How many cells do we have now in our dataset?
>
> Print the **shape** of your gene expression matrix.
>

<details>
<summary><i>Show solution</i></summary>

```python
print(adata.shape)      
```
    
We now have 61724 cells.

</details>

## 4) Data normalization
We first need to store the raw counts on a different slot, we will call it "counts".

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
adata.layers["counts"][:10, :10].toarray()

Now we perform library size normalization.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)

How does it look like?

In [ ]:
adata.X[:10, :10].toarray()

Now let's log transform the data.

In [ ]:
sc.pp.log1p(adata)

How does it look like?

In [ ]:
adata.X[:10, :10].toarray()

> ### ✏️ Exercise 3 — Let's now make the same histogram of expression values as before, do you see any changes?
> Set log=False

<details>
<summary><i>Show solution</i></summary>

```python
plt.hist(adata.X.data, bins=50, log=False)
plt.ticklabel_format(style='plain', axis='y')
plt.xlabel("Normalized expression value")
plt.ylabel("Frequency")
plt.show() 
```

</details>

## 5) Feature selection - Highly variable genes
Let's identify the top 2000 highly variable genes.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    flavor="seurat_v3",
    layer="counts",
    batch_key="Dataset",
    subset=False,         # keep all genes in the object
)

Where are they stored?

In [ ]:
adata.var

In [ ]:
# —— PLOT MEAN VS NORMALISED VARIANCE ————————————————————————————
hv = adata.var["highly_variable"].values

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(adata.var.loc[~hv, "means"], adata.var.loc[~hv, "variances_norm"],
           s=3, c="lightgrey", label="other genes")
ax.scatter(adata.var.loc[hv, "means"], adata.var.loc[hv, "variances_norm"],
           s=3, c="#DD8452", label=f"HVG (n={hv.sum()})")
ax.set_xscale("log")                     # mean expression spans orders of magnitude
ax.set_xlabel("mean expression (log scale)")
ax.set_ylabel("normalised variance")
ax.legend(frameon=False)
ax.set_title("Highly variable gene selection")
plt.tight_layout()
plt.show()

Which are the top 10 highly variable genes?

In [ ]:
top = (adata.var.loc[hv]
       .sort_values("variances_norm", ascending=False)
       .head(10)[["means", "variances_norm"]])
display(top)

## 6) Saving the object
Finally, let's save the object.

In [ ]:
adata.write_h5ad(OUT_DIR + "adata_s2part1.h5ad")